# Test-Driven Development Walkthrough

Test-Driven Development, usually called **TDD**, is a way of building software in very small steps. Instead of writing a large amount of code first and testing later, we begin by thinking about the behavior we want, write a test for that behavior, and then write just enough code to make the test pass.

The classic TDD cycle is:

1. **Red**: write a test for behavior that does not exist yet
2. **Green**: write the smallest amount of code needed to pass the test
3. **Refactor**: clean up the code while keeping the tests passing

In this notebook, we will build a small `BankAccount` class step by step. Each section follows the same rhythm:

- a markdown cell that explains the goal
- a code cell that implements the step and runs tests
- a markdown cell that explains the implementation details

This makes the development process visible and helps students see that testing is not something we add at the end. It is part of the design process from the beginning.

## Goal 1: Create a Bank Account with a Starting Balance

Our first goal is very small. We want to create a `BankAccount` object that stores an owner's name and a starting balance. In TDD terms, this is our first tiny piece of behavior.

In [1]:
import unittest


def run_test_case(test_case):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_case)
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)


class BankAccount:
    def __init__(self, owner, starting_balance=0):
        self.owner = owner
        self.balance = starting_balance


class TestBankAccountStep1(unittest.TestCase):
    def test_account_stores_owner_name(self):
        account = BankAccount("Ava", 100)
        self.assertEqual(account.owner, "Ava")

    def test_account_stores_starting_balance(self):
        account = BankAccount("Ava", 100)
        self.assertEqual(account.balance, 100)


run_test_case(TestBankAccountStep1)

test_account_stores_owner_name (__main__.TestBankAccountStep1.test_account_stores_owner_name) ... ok
test_account_stores_starting_balance (__main__.TestBankAccountStep1.test_account_stores_starting_balance) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


### Implementation Notes for Goal 1

This is a very small start, and that is exactly the point. TDD encourages us to avoid building too much at once.

A few details matter here:

- The class only does what the tests currently require.
- We check one behavior at a time with small test methods.
- The tests act like a written description of the class's expected state.

Even at this early stage, the tests give us confidence that object creation works the way we intended.

## Goal 2: Add a Deposit Method

Now we want the account to support deposits. The new behavior is simple: when money is deposited, the balance should increase by that amount.

In [2]:
class BankAccount:
    def __init__(self, owner, starting_balance=0):
        self.owner = owner
        self.balance = starting_balance

    def deposit(self, amount):
        self.balance += amount


class TestBankAccountStep2(unittest.TestCase):
    def test_account_stores_starting_balance(self):
        account = BankAccount("Ava", 100)
        self.assertEqual(account.balance, 100)

    def test_deposit_increases_balance(self):
        account = BankAccount("Ava", 100)
        account.deposit(25)
        self.assertEqual(account.balance, 125)


run_test_case(TestBankAccountStep2)

test_account_stores_starting_balance (__main__.TestBankAccountStep2.test_account_stores_starting_balance) ... ok
test_deposit_increases_balance (__main__.TestBankAccountStep2.test_deposit_increases_balance) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.002s

OK


### Implementation Notes for Goal 2

Notice that we did not throw away our earlier idea. We kept the original behavior and added one small new feature.

That is a key TDD habit:

- keep the old behavior working
- add one new behavior
- run the tests again

The `deposit` method is intentionally simple. It only changes the balance. At this point, we are not adding validation yet because our tests have not asked for that behavior. TDD helps us avoid solving problems too early.

## Goal 3: Add a Withdraw Method

Next, we want to withdraw money from the account. The balance should go down when a valid withdrawal happens.

In [5]:
class BankAccount:
    def __init__(self, owner, starting_balance=0):
        self.owner = owner
        self.balance = starting_balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        self.balance -= amount


class TestBankAccountStep3(unittest.TestCase):
    def setUp(self):
        self.account = BankAccount("Ava", 100)

    def test_deposit_increases_balance(self):
        self.account.deposit(25)
        self.assertEqual(self.account.balance, 125)

    def test_withdraw_decreases_balance(self):
        self.account.withdraw(30)
        self.assertEqual(self.account.balance, 70)


run_test_case(TestBankAccountStep3)

test_deposit_increases_balance (__main__.TestBankAccountStep3.test_deposit_increases_balance) ... ok
test_withdraw_decreases_balance (__main__.TestBankAccountStep3.test_withdraw_decreases_balance) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.002s

OK


### Implementation Notes for Goal 3

This step introduces `setUp()`, which is a very common testing tool in `unittest`. The `setUp()` method runs before each test method, so every test starts with a fresh `BankAccount` object.

That matters because tests should be independent. If one test changes the balance, another test should not accidentally inherit that changed state.

This section also shows an important TDD idea: we are gradually growing the class. Each step is still small enough to understand, test, and fix quickly if something breaks.

## Goal 4: Prevent Invalid Withdrawals

A real bank account should not allow someone to withdraw more money than is available. Now our tests need to describe not only correct behavior, but also behavior that should be rejected.

In [6]:
class BankAccount:
    def __init__(self, owner, starting_balance=0):
        self.owner = owner
        self.balance = starting_balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Cannot withdraw more than the current balance.")

        self.balance -= amount


class TestBankAccountStep4(unittest.TestCase):
    def setUp(self):
        self.account = BankAccount("Ava", 100)

    def test_valid_withdraw_decreases_balance(self):
        self.account.withdraw(30)
        self.assertEqual(self.account.balance, 70)

    def test_withdrawing_too_much_raises_error(self):
        with self.assertRaises(ValueError):
            self.account.withdraw(150)

    def test_failed_withdraw_does_not_change_balance(self):
        try:
            self.account.withdraw(150)
        except ValueError:
            pass

        self.assertEqual(self.account.balance, 100)


run_test_case(TestBankAccountStep4)

test_failed_withdraw_does_not_change_balance (__main__.TestBankAccountStep4.test_failed_withdraw_does_not_change_balance) ... ok
test_valid_withdraw_decreases_balance (__main__.TestBankAccountStep4.test_valid_withdraw_decreases_balance) ... ok
test_withdrawing_too_much_raises_error (__main__.TestBankAccountStep4.test_withdrawing_too_much_raises_error) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### Implementation Notes for Goal 4

This is an important TDD moment because we are testing a rule, not just a calculation.

Two good testing ideas appear here:

- `assertRaises(ValueError)` checks that invalid behavior is rejected on purpose.
- We also test that a failed withdrawal does not secretly damage the object's state.

That second test is subtle but valuable. It shows students that tests can check both the obvious outcome and the hidden side effects. Good tests help us protect behavior, not just produce nice-looking output.

## Goal 5: Refactor by Adding a Balance Checker Method

TDD is not only about adding features. It also includes refactoring. Now that our class is working, we can improve the design a little by giving it a method that returns the current balance in a clean, readable way.

In [ ]:
class BankAccount:
    def __init__(self, owner, starting_balance=0):
        self.owner = owner
        self.balance = starting_balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Cannot withdraw more than the current balance.")

        self.balance -= amount

    def get_balance(self):
        return self.balance


class TestBankAccountStep5(unittest.TestCase):
    def setUp(self):
        self.account = BankAccount("Ava", 100)

    def test_get_balance_returns_current_balance(self):
        self.assertEqual(self.account.get_balance(), 100)

    def test_get_balance_after_deposit(self):
        self.account.deposit(40)
        self.assertEqual(self.account.get_balance(), 140)

    def test_get_balance_after_withdraw(self):
        self.account.withdraw(25)
        self.assertEqual(self.account.get_balance(), 75)


run_test_case(TestBankAccountStep5)

### Implementation Notes for Goal 5

This final step highlights the **refactor** part of TDD. Once the core behavior is protected by tests, we can improve the design with less fear.

The `get_balance()` method is small, but it gives the class a cleaner public interface. In larger programs, this kind of change matters because it helps separate how the object stores data from how other code interacts with it.

The larger lesson is this: tests are not just for catching bugs. They also make it safer to improve code over time.

## Closing Thoughts

This notebook walked through TDD as a sequence of small, manageable steps. We started with object creation, added behavior one piece at a time, tested valid and invalid actions, and then made a small refactor.

That is the heart of Test-Driven Development:

- think about behavior first
- write a test for that behavior
- write the smallest amount of code to pass the test
- improve the design while keeping the tests green

If students learn to work in small, tested steps, they usually become more confident programmers because the computer gives them quick feedback as they build.